# Lab 6.2 &mdash; The Docstring Is the Tool's API

**Level:** Beginner &nbsp;|&nbsp; **Est. time:** 20 min &nbsp;|&nbsp; **Day 3 &middot; Module 6 &mdash; Frameworks for Building AI Agents**

### What you'll do
- Write descriptions that tell the model WHEN to use each tool
- Bind the tools to a real agent and ask a targeted question
- Read the trace to see the description decide which tool fires

> **How this lab works (near-real):** you have a local Ollama running `llama3.1:8b`. Read the **Concept**, fill the real `___` blanks in **Build it** (real tool bodies, real prompts, the real `create_agent` call), then **Run it for real** &mdash; a real LLM drives a real agent over real tools &mdash; and **read the trace** to see exactly what it did. Finish with an open **Your turn**. There is **no auto-grader**; the goal is a working agent and a trace you can read.

> **Framework note:** these labs use the **real** LangChain (`langchain`, `langchain-core`, `langchain-ollama`, `langgraph`) and a **real local model** (`ollama run llama3.1:8b`, pinned to `http://127.0.0.1:11434`). If Ollama is down, the run cells print how to start it instead of crashing. A `@tool` must **catch its own errors and return a string** &mdash; a tool that *raises* aborts the whole agent run (you'll see exactly this in Lab 11).

**Reference:** [Module 6 slides &mdash; Defining a tool](../../presentation/day3-module6-frameworks-for-building-ai-agents.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 6 labs](./index.html)

In [ ]:
# Setup -- run me first
import os, socket, pathlib
from dotenv import load_dotenv
load_dotenv(pathlib.Path("/home/rajesh/Training/courses/building-intelligents-ai-agents/.env"))   # SERPER_API_KEY, WOLFRAM_ALPHA_APPID, GROQ/OPENAI keys

WORK = "/tmp/biaa-lab-06-02"
os.makedirs(WORK, exist_ok=True)

def ollama_up(host="127.0.0.1", port=11434):
    """True if a local Ollama server is listening. If it's down, start it with:  ollama run llama3.1:8b"""
    try:
        with socket.create_connection((host, port), timeout=1):
            return True
    except OSError:
        return False

from langchain_ollama import ChatOllama
# Day-3 provider: a REAL local model. Pin the host -- plain 'localhost' can give 'No route to host'.
llm = ChatOllama(model="llama3.1:8b", temperature=0, base_url="http://127.0.0.1:11434")

def print_trace(result):
    """Print a REAL agent message trace: tool calls the model made, tool observations, final answer."""
    for m in result["messages"]:
        for tc in (getattr(m, "tool_calls", None) or []):
            print("TOOL CALL:", tc["name"], tc["args"])
        if type(m).__name__ == "ToolMessage":
            print("OBS:", str(m.content)[:200])
        elif str(getattr(m, "content", "")).strip():
            print(type(m).__name__, ":", str(m.content)[:300])

if ollama_up():
    print("Ollama reachable at 127.0.0.1:11434 | model:", llm.model, "| WORK:", WORK)
else:
    print("Ollama NOT reachable -- start it with:  ollama run llama3.1:8b")
    print("(The 'Run it for real' cells will print this note instead of crashing.)  WORK:", WORK)

## Concept
The model never sees your code &mdash; only each tool's **name**, **description** and **args**. So the
**docstring is the tool's real API**: a vague one makes the agent pick the wrong tool (or none). Here you
write good descriptions, bind the tools to a real agent, and **watch the trace** confirm the model routed
by your words.

In [ ]:
from langchain_core.tools import tool

@tool
def clock(_: str) -> str:
    """Return the current time of day."""
    return "12:00"

print("the model is shown -> ", clock.name + ": " + clock.description)

## Build it
Write three tools with **clear descriptions** (that's the part graded by reality &mdash; a good agent
run). The bodies are simple; the **docstring** is what matters.

In [ ]:
from langchain_core.tools import tool

@tool
def weather(city: str) -> str:
    """___  (TODO: tell the model WHEN to use this -- e.g. current temperature/conditions for a city)."""
    return city.title() + ": sunny, 24C"

@tool
def dictionary(word: str) -> str:
    """___  (TODO: tell the model WHEN to use this -- e.g. the meaning of an English word)."""
    return word + ": (a definition)"

@tool
def translate(text: str) -> str:
    """___  (TODO: tell the model WHEN to use this -- e.g. translate text into French)."""
    return "(translated) " + text

TOOLS = [weather, dictionary, translate]

try:
    print("catalog the model is shown:")
    for t in TOOLS:
        print(" -", t.name + ":", t.description)
except Exception as e:
    print("(Fill the ___ blanks above, then re-run.)", type(e).__name__)

## Run it for real &amp; read the trace
Bind the three tools to a real agent and ask a weather question. The description &mdash; not any keyword you coded &mdash; is what makes the model pick `weather`.

_This calls the real `llama3.1:8b` via Ollama. If Ollama is down the cell prints how to start it instead of crashing._

In [ ]:
if not ollama_up():
    print("Ollama not reachable -- start `ollama run llama3.1:8b`, then re-run this cell.")
else:
    try:
        from langchain.agents import create_agent
        agent = create_agent(llm, tools=TOOLS)
        result = agent.invoke({"messages": [("user", "What's the weather like in Tokyo right now?")]},
                              config={"recursion_limit": 6})
        print_trace(result)
    except Exception as e:
        print("(Fill the ___ blanks above, then re-run.)", type(e).__name__)

## What to notice
- The trace should show **`TOOL CALL: weather {'city': 'Tokyo'}`** &mdash; chosen from your description, not your code.
- The model matched the *question* to the *docstring*. That is the entire selection mechanism.

## Your turn (open task &mdash; no grader)
Change `weather`'s docstring to something vague like `"""Does stuff with a place."""` and re-run the agent
on the same question. **What good looks like:** with a vague description the model often stops calling
`weather` (or calls the wrong tool). Restore a clear description and it comes back. You've just proven the
docstring is the API &mdash; write it for the model.

---
### Key takeaway
The description is what the model reads to choose a tool -- write it for the model. You watched a real agent route by your words, and watched a vague docstring break it.

[&#8592; All Module 6 labs](./index.html) &nbsp;&middot;&nbsp; [Module 6 slides](../../presentation/day3-module6-frameworks-for-building-ai-agents.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>